# Real-Time Object Detection with YOLO and TensorRT Optimization

This notebook demonstrates how to optimize YOLO models using NVIDIA TensorRT for high-performance real-time object detection.

## Features:
- Custom YOLO model loading (yolo26x.pt)
- TensorRT engine conversion and optimization
- Real-time inference on images and videos
- Performance benchmarking


## Step 1: Install Dependencies and Setup


In [ ]:
# Check GPU availability
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

# Install required packages
!pip install ultralytics onnx onnxsim tensorrt --quiet
!pip install opencv-python-headless --quiet


## Step 2: Import Libraries


In [ ]:
import os
import cv2
import numpy as np
import torch
from pathlib import Path
from ultralytics import YOLO
import time
from typing import List, Tuple
import warnings
warnings.filterwarnings('ignore')

# Create directories
os.makedirs('models', exist_ok=True)
os.makedirs('tensorrt_engines', exist_ok=True)
os.makedirs('outputs', exist_ok=True)


## Step 3: Load Custom YOLO Model

Load your custom YOLO model from the specified path. If the model is not found, you can upload it using the cell below.


In [ ]:
# Optional: Upload model file if not already in Colab
# Uncomment the lines below if you need to upload your model

# from google.colab import files
# uploaded = files.upload()
# # After upload, move to /content/ if needed
# import shutil
# for filename in uploaded.keys():
#     if filename.endswith('.pt'):
#         shutil.move(filename, '/content/yolo26x.pt')
#         print(f"Model moved to /content/yolo26x.pt")


In [ ]:
# Load custom YOLO model from path
model_path = '/content/yolo26x.pt'  # Custom model path

# Check if model exists
if not os.path.exists(model_path):
    print(f"⚠ Warning: Model not found at {model_path}")
    print("Please upload your model to Colab or update the model_path variable")
    print("You can upload files using: from google.colab import files; files.upload()")
else:
    print(f"Loading model from: {model_path}")
    model = YOLO(model_path)
    
    # Extract model name from path for naming outputs
    model_name = os.path.splitext(os.path.basename(model_path))[0]  # 'yolo26x'
    
    print(f"✓ Model loaded: {model_name}")
    print(f"✓ Model classes: {len(model.names)} classes")
    print(f"✓ Model device: {model.device}")
    print(f"✓ Model path: {model_path}")


## Step 4: Export Model to ONNX Format


In [ ]:
# Export YOLO model to ONNX format (required for TensorRT conversion)
onnx_path = f'models/{model_name}.onnx'
print(f"Exporting model to ONNX format...")
model.export(format='onnx', imgsz=640, simplify=True)
print(f"✓ Model exported to ONNX: {onnx_path}")


## Step 5: Convert ONNX to TensorRT Engine


### Alternative: Fixed Manual TensorRT Conversion (if Step 6 doesn't work)


In [ ]:
# Fixed TensorRT conversion function (for TensorRT 8.5+)
# Only use this if the Ultralytics export in Step 6 doesn't work

import tensorrt as trt

def build_engine_fixed(onnx_path, engine_path, precision='fp16'):
    """Fixed version for TensorRT 8.5+ API"""
    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    parser = trt.OnnxParser(network, logger)
    
    # Parse ONNX model
    with open(onnx_path, 'rb') as model_file:
        if not parser.parse(model_file.read()):
            for error in range(parser.num_errors):
                print(parser.get_error(error))
            return None
    
    # Configure builder - FIXED for TensorRT 8.5+
    config = builder.create_builder_config()
    
    # Use set_memory_pool_limit for TensorRT 8.5+ (instead of max_workspace_size)
    try:
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)  # 1GB
    except (AttributeError, TypeError):
        # Fallback for older versions
        try:
            config.max_workspace_size = 1 << 30
        except:
            pass
    
    # Set precision
    if precision == 'fp16' and builder.platform_has_fast_fp16:
        config.set_flag(trt.BuilderFlag.FP16)
        print("Using FP16 precision")
    elif precision == 'int8' and builder.platform_has_fast_int8:
        config.set_flag(trt.BuilderFlag.INT8)
        print("Using INT8 precision")
    else:
        print("Using FP32 precision")
    
    # Build engine
    print("Building TensorRT engine... This may take a few minutes.")
    engine = builder.build_engine(network, config)
    
    if engine is None:
        print("Failed to build engine")
        return None
    
    # Save engine
    with open(engine_path, 'wb') as f:
        f.write(engine.serialize())
    
    print(f"✓ TensorRT engine saved to: {engine_path}")
    return engine

# Uncomment to use fixed version:
# precision = 'fp16'
# engine_path = f'tensorrt_engines/{model_name}_{precision}.engine'
# onnx_model_path = f'{model_name}.onnx'
# build_engine_fixed(onnx_model_path, engine_path, precision)


In [ ]:
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit

def build_engine(onnx_path, engine_path, precision='fp16'):
    """
    Convert ONNX model to TensorRT engine
    
    Args:
        onnx_path: Path to ONNX model
        engine_path: Path to save TensorRT engine
        precision: 'fp32', 'fp16', or 'int8'
    """
    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    parser = trt.OnnxParser(network, logger)
    
    # Parse ONNX model
    with open(onnx_path, 'rb') as model:
        if not parser.parse(model.read()):
            for error in range(parser.num_errors):
                print(parser.get_error(error))
            return None
    
    # Configure builder
    config = builder.create_builder_config()
    config.max_workspace_size = 1 << 30  # 1GB
    
    # Set precision
    if precision == 'fp16' and builder.platform_has_fast_fp16:
        config.set_flag(trt.BuilderFlag.FP16)
        print("Using FP16 precision")
    elif precision == 'int8' and builder.platform_has_fast_int8:
        config.set_flag(trt.BuilderFlag.INT8)
        print("Using INT8 precision")
    else:
        print("Using FP32 precision")
    
    # Build engine
    print("Building TensorRT engine... This may take a few minutes.")
    engine = builder.build_engine(network, config)
    
    if engine is None:
        print("Failed to build engine")
        return None
    
    # Save engine
    with open(engine_path, 'wb') as f:
        f.write(engine.serialize())
    
    print(f"TensorRT engine saved to: {engine_path}")
    return engine

# Convert to TensorRT
precision = 'fp16'  # Change to 'fp32' or 'int8' if needed
engine_path = f'tensorrt_engines/{model_name}_{precision}.engine'
onnx_model_path = f'{model_name}.onnx'  # Ultralytics saves in current directory

if not os.path.exists(engine_path):
    engine = build_engine(onnx_model_path, engine_path, precision)
else:
    print(f"Engine already exists: {engine_path}")


## Step 6: Load TensorRT Engine and Create Inference Function


In [ ]:
# Export YOLO model to TensorRT engine using Ultralytics (simpler method)
try:
    # Export to TensorRT format using Ultralytics built-in method
    engine_path = model.export(format='engine', imgsz=640, half=True)  # half=True for FP16
    print(f"✓ TensorRT engine created successfully: {engine_path}")
    
    # Load the TensorRT engine
    model_trt = YOLO(engine_path)
    print("✓ TensorRT model loaded and ready for inference!")
except Exception as e:
    print(f"⚠ TensorRT export error: {e}")
    print("Using PyTorch backend instead")
    model_trt = model


## Step 7: Image Inference


In [ ]:
# Run inference on an image
# You can upload an image or use a sample image
from google.colab import files
from IPython.display import display
from PIL import Image as PILImage  # Import PIL Image for fromarray

# Option 1: Upload your own image
uploaded = files.upload()
image_path = list(uploaded.keys())[0] if uploaded else None

# Option 2: Use a sample image URL
if not image_path:
    import urllib.request
    sample_url = "https://ultralytics.com/images/bus.jpg"
    image_path = "sample_image.jpg"
    urllib.request.urlretrieve(sample_url, image_path)
    print("Using sample image from Ultralytics")

# Run inference
print(f"Running inference on: {image_path}")
results = model_trt(image_path)

# Display results
for r in results:
    im_array = r.plot()  # plot a BGR numpy array of predictions
    im = PILImage.fromarray(im_array[..., ::-1])  # RGB PIL image - fixed import
    display(im)  # show image
    
    # Print detection results
    print(f"\nDetections: {len(r.boxes)} objects found")
    for box in r.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        print(f"  - {model_trt.names[cls]}: {conf:.2f}")

# Save result
output_path = f"outputs/detection_{os.path.basename(image_path)}"
results[0].save(output_path)
print(f"\n✓ Result saved to: {output_path}")


## Step 8: Video Processing


In [ ]:
# Process video file
from google.colab import files

# Upload video file
print("Please upload a video file:")
uploaded_video = files.upload()
video_path = list(uploaded_video.keys())[0] if uploaded_video else None

if video_path:
    output_video = f"outputs/detected_{os.path.basename(video_path)}"
    
    print(f"Processing video: {video_path}")
    print("This may take a few minutes...")
    
    # Run inference on video
    results = model_trt.predict(
        source=video_path,
        save=True,
        save_txt=False,
        conf=0.25,
        project='outputs',
        name='video_detection'
    )
    
    print(f"✓ Video processed and saved!")
    print(f"✓ Output: {output_video}")
    
    # Display video
    from IPython.display import HTML
    from base64 import b64encode
    
    mp4 = open(output_video, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    HTML(f"""
    <video width=640 controls>
      <source src="{data_url}" type="video/mp4">
    </video>
    """)
else:
    print("No video uploaded. Skipping video processing.")


## Step 9: Performance Benchmarking


In [ ]:
# Benchmark inference speed
import time

# Create a dummy image for benchmarking
dummy_image = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
cv2.imwrite("dummy_test.jpg", dummy_image)

# Warmup
print("Warming up...")
for _ in range(10):
    _ = model_trt("dummy_test.jpg", verbose=False)

# Benchmark PyTorch model
print("\nBenchmarking PyTorch model...")
times_pytorch = []
for i in range(50):
    start = time.time()
    _ = model("dummy_test.jpg", verbose=False)
    times_pytorch.append(time.time() - start)
    if (i + 1) % 10 == 0:
        print(f"  Completed {i+1}/50 iterations")

# Benchmark TensorRT model
print("\nBenchmarking TensorRT model...")
times_tensorrt = []
for i in range(50):
    start = time.time()
    _ = model_trt("dummy_test.jpg", verbose=False)
    times_tensorrt.append(time.time() - start)
    if (i + 1) % 10 == 0:
        print(f"  Completed {i+1}/50 iterations")

# Calculate statistics
pytorch_fps = 1.0 / np.mean(times_pytorch)
tensorrt_fps = 1.0 / np.mean(times_tensorrt)
speedup = tensorrt_fps / pytorch_fps

print("\n" + "="*50)
print("PERFORMANCE RESULTS")
print("="*50)
print(f"PyTorch Model:")
print(f"  Average FPS: {pytorch_fps:.2f}")
print(f"  Average Latency: {np.mean(times_pytorch)*1000:.2f} ms")
print(f"\nTensorRT Model:")
print(f"  Average FPS: {tensorrt_fps:.2f}")
print(f"  Average Latency: {np.mean(times_tensorrt)*1000:.2f} ms")
print(f"\n🚀 Speedup: {speedup:.2f}x faster with TensorRT!")
print("="*50)


## Step 10: Real-Time Webcam Detection (Colab-Compatible)

This version works in Google Colab using the camera widget. For local use, see the alternative function below.


In [ ]:
# Colab-compatible real-time detection using camera widget
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode, b64encode
from PIL import Image as PILImage  # Import PIL Image
import io
import html

def take_photo(filename='photo.jpg', quality=0.8):
    """Capture photo from Colab camera"""
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            const capture = document.createElement('button');
            capture.textContent = 'Capture Photo';
            div.appendChild(capture);

            const video = document.createElement('video');
            video.style.display = 'block';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();

            // Resize the output to fit the video element.
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

            // Wait for Capture to be clicked.
            await new Promise((resolve) => {
                capture.onclick = resolve;
            });

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

def real_time_detection_colab(conf_threshold=0.25, num_frames=10):
    """
    Real-time detection in Colab using camera widget
    
    Args:
        conf_threshold: Confidence threshold for detections
        num_frames: Number of frames to capture and process
    """
    fps_list = []
    
    print("Starting real-time detection in Colab...")
    print(f"Will capture {num_frames} frames")
    print("Click 'Capture Photo' button when ready")
    
    for frame_num in range(num_frames):
        print(f"\n--- Frame {frame_num + 1}/{num_frames} ---")
        
        # Capture photo
        photo_path = f'temp_frame_{frame_num}.jpg'
        try:
            take_photo(photo_path)
        except Exception as e:
            print(f"Error capturing photo: {e}")
            print("Make sure to allow camera access when prompted")
            break
        
        # Run inference
        start_time = time.time()
        results = model_trt.predict(
            source=photo_path,
            conf=conf_threshold,
            verbose=False
        )
        inference_time = time.time() - start_time
        
        # Calculate FPS
        fps = 1.0 / inference_time
        fps_list.append(fps)
        
        # Draw results and display
        annotated_frame = results[0].plot()
        
        # Add FPS text
        cv2.putText(annotated_frame, f'FPS: {fps:.1f}', (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(annotated_frame, f'Frame: {frame_num + 1}/{num_frames}', (10, 70),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        # Save and display
        output_path = f'outputs/frame_{frame_num + 1}.jpg'
        cv2.imwrite(output_path, annotated_frame)
        
        # Display image (convert BGR to RGB for PIL)
        annotated_frame_rgb = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)
        im = PILImage.fromarray(annotated_frame_rgb)
        display(im)
        
        print(f"Frame {frame_num + 1}: {fps:.2f} FPS, {len(results[0].boxes)} detections")
        
        # Clean up temp file
        if os.path.exists(photo_path):
            os.remove(photo_path)
    
    if fps_list:
        print("\n" + "="*50)
        print("PERFORMANCE SUMMARY")
        print("="*50)
        print(f"Average FPS: {np.mean(fps_list):.2f}")
        print(f"Min FPS: {np.min(fps_list):.2f}")
        print(f"Max FPS: {np.max(fps_list):.2f}")
        print("="*50)

# Run real-time detection in Colab
# Uncomment to run:
# real_time_detection_colab(conf_threshold=0.25, num_frames=5)

# Alternative: Process video file frame by frame (works better in Colab)
def process_video_realtime(video_path, conf_threshold=0.25, max_frames=None):
    """
    Process video file frame by frame with real-time-like display
    
    Args:
        video_path: Path to video file
        conf_threshold: Confidence threshold
        max_frames: Maximum number of frames to process (None for all)
    """
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"Error: Could not open video: {video_path}")
        return
    
    fps_list = []
    frame_count = 0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"Processing video: {video_path}")
    print(f"Total frames: {total_frames}")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        if max_frames and frame_count >= max_frames:
            break
        
        # Run inference
        start_time = time.time()
        results = model_trt.predict(
            source=frame,
            conf=conf_threshold,
            verbose=False
        )
        inference_time = time.time() - start_time
        
        # Calculate FPS
        fps = 1.0 / inference_time
        fps_list.append(fps)
        
        # Draw results
        annotated_frame = results[0].plot()
        cv2.putText(annotated_frame, f'FPS: {fps:.1f}', (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(annotated_frame, f'Frame: {frame_count}/{total_frames}', (10, 70),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        # Display every 10th frame to avoid too many outputs
        if frame_count % 10 == 0:
            im = PILImage.fromarray(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
            display(im)
            print(f"Frame {frame_count}: {fps:.2f} FPS, {len(results[0].boxes)} detections")
        
        frame_count += 1
    
    cap.release()
    
    if fps_list:
        print("\n" + "="*50)
        print("VIDEO PROCESSING SUMMARY")
        print("="*50)
        print(f"Frames processed: {frame_count}")
        print(f"Average FPS: {np.mean(fps_list):.2f}")
        print(f"Min FPS: {np.min(fps_list):.2f}")
        print(f"Max FPS: {np.max(fps_list):.2f}")
        print("="*50)

# Example: Process uploaded video
# Uncomment and upload a video first:
# process_video_realtime('your_video.mp4', max_frames=100)


## Summary

This notebook demonstrated:
- ✅ Custom YOLO model loading (yolo26x.pt)
- ✅ Model export to ONNX format
- ✅ TensorRT engine conversion and optimization
- ✅ Image inference with object detection
- ✅ Video processing capabilities
- ✅ Performance benchmarking (PyTorch vs TensorRT)

### Key Benefits of TensorRT:
- **Faster Inference**: 2-5x speedup compared to PyTorch
- **Lower Latency**: Reduced processing time per frame
- **Optimized Memory**: Efficient GPU memory usage
- **Production Ready**: Optimized for deployment

### Next Steps:
- Experiment with different precision modes (FP32, FP16, INT8)
- Process your own images and videos
- Deploy the optimized model to production
- Try different model sizes or architectures
